In [4]:
!apt-get -y install tesseract-ocr
!pip install gradio pytesseract opencv-python pillow google-generativeai SpeechRecognition gtts

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
  Using cached speechrecognition-3.16.1-py3-none-any.whl.metadata (28 kB)
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)
Using cached speechrecognition-3.16.1-py3-none-any.whl (32.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
  Attempting uninstall: typer
    Found existing installation: type

In [5]:
import gradio as gr
from PIL import Image
import speech_recognition as sr
import google.generativeai as genai
import re, os, base64, json, io
from datetime import datetime, timedelta
import pytesseract
import cv2
import numpy as np

genai.configure(api_key="YOUR_API_KEY")
saved_reminders = []

def compress_image(img, max_size=800):
    """Resize image. Keep buffer alive so PIL does not lose data."""
    w, h = img.size
    if max(w, h) > max_size:
        ratio = max_size / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    img = img.convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=75)
    buf.seek(0)
    result = Image.open(buf)
    result.load()
    return result

def ocr_image_text(img):
    try:
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, None, fx=2, fy=2)
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 11, 2
        )
        return pytesseract.image_to_string(thresh, config='--psm 6')
    except:
        return ""

def is_text_useful(text):
    if not text:
        return False
    clean = re.sub(r'\s+', '', text)
    if len(clean) < 20:
        return False
    letters = sum(c.isalpha() for c in clean)
    ratio = letters / len(clean)
    return ratio > 0.4

def speech_to_text(audio_file):
    try:
        r = sr.Recognizer()
        with sr.AudioFile(audio_file) as source:
            audio = r.record(source)
        return r.recognize_google(audio)
    except Exception as e:
        print("STT error:", e)
        return ""

def make_calendar_html(title, date, time_str):
    if not date:
        return """
        <div style="
            background:#FFEBEE;
            border:2px solid #EF9A9A;
            border-radius:16px;
            padding:18px;
            color:#B71C1C;
            font-size:18px;
            font-weight:800;">
            ⚠️ No date found.
        </div>
        """
    if not time_str:
        time_str = "09:00"
    return f"""
    <div style="
        background:#E8F5E9;
        border:2px solid #A5D6A7;
        border-radius:20px;
        padding:22px;
        margin-top:12px;
        box-shadow:0 4px 12px rgba(46,125,50,0.15);">
        <div style="
            font-size:24px;
            font-weight:900;
            color:#1A237E;
            margin-bottom:12px;">
            {title if title else "Reminder"}
        </div>
        <div style="
            font-size:18px;
            color:#1B5E20;
            font-weight:700;
            line-height:1.8;">
            🗓️ Date: {date}<br>
            ⏰ Time: {time_str}
        </div>
        <div style="
            margin-top:14px;
            background:#C8E6C9;
            padding:10px 14px;
            border-radius:12px;
            color:#1B5E20;
            font-size:16px;
            font-weight:700;">
            ✔ Reminder stored successfully
        </div>
    </div>
    """

# ── LANGUAGE-AWARE PROMPTS ──────────────────────────────────────

LANG_INSTRUCTION = {
    "English": "Respond ONLY in English.",
    "Kannada": "Respond ONLY in Kannada (ಕನ್ನಡ). Use simple, everyday Kannada words suitable for elderly people.",
    "Hindi":   "Respond ONLY in Hindi (हिंदी). Use simple, everyday Hindi words suitable for elderly people.",
}

def get_prompt(lang="English"):
    lang_note = LANG_INSTRUCTION.get(lang, LANG_INSTRUCTION["English"])
    return f"""
You are an AI assistant helping elderly users safely understand documents.

{lang_note}

STEP 1:
Check if the document could be dangerous, fraudulent, misleading, or suspicious.

Treat these as HIGH RISK:
- OTP/password requests
- urgent payment threats
- fake bank/account alerts
- suspicious links
- lottery/prize scams
- requests for money or verification

If uncertain or partially suspicious,
DO NOT mark safe.
Prefer:
- "Needs Attention"
- higher risk
- suspicious classification

STEP 2:
If safe, classify into:
Medicine,
Appointment,
Bill,
Bank,
Government,
Delivery,
Invitation,
Event,
Shopping,
Travel,
Emergency,
General.

STEP 3:
Explain in SIMPLE language for a senior citizen.
Write the explanation and action fields in {lang}.

Return ONLY valid JSON:
{{
 "category":"",
 "is_scam":false,
 "risk":0-100,
 "explanation":"",
 "action":"",
 "date":"",
 "time":""
}}

Senior safety is the top priority.

A false "Looks Safe" result is dangerous.

If the content contains:
- links
- urgency
- payment pressure
- verification requests
- threats
- unknown QR codes

increase risk significantly.
"""

# ── MULTILINGUAL KEYWORD CLASSIFIER ──────────────────────────────

LANG_LABELS = {
    "English": {
        "scam_explanation": "This message looks suspicious and may be a scam or phishing attempt.",
        "scam_action": "Do NOT click links, scan QR codes, or share personal information.",
        "medicine_explanation": "This appears to contain medicine or prescription instructions.",
        "medicine_action": "Follow the doctor's advice carefully.",
        "bill_explanation": "This appears to be a bill or payment-related document.",
        "bill_action": "Check the amount and due date carefully.",
        "delivery_explanation": "This appears to be a delivery or shipment update.",
        "delivery_action": "Track the package if needed.",
        "appointment_explanation": "This appears to be an appointment or scheduled visit.",
        "appointment_action": "Remember the date and time.",
        "invitation_explanation": "This appears to be an invitation or event notice.",
        "invitation_action": "Check the event details and timing.",
        "bank_explanation": "This appears to be a banking or transaction message.",
        "bank_action": "Review the transaction details carefully.",
        "shopping_explanation": "This appears to be a shopping-related message.",
        "shopping_action": "Review the order details.",
        "travel_explanation": "This appears to be a travel or ticket-related message.",
        "travel_action": "Check your travel schedule carefully.",
        "general_explanation": "This is a general informational document.",
        "general_action": "Please read carefully or ask a family member if unsure.",
        "error_explanation": "Could not connect to AI. Check your API key and internet.",
        "error_action": "Please try again or check your GEMINI_API_KEY.",
        "no_input_explanation": "Please provide an input.",
        "no_input_action": "Try again.",
        "stt_error": "Could not hear clearly.",
        "stt_retry": "Please try again.",
    },
    "Kannada": {
        "scam_explanation": "ಈ ಸಂದೇಶವು ಅನುಮಾನಾಸ್ಪದವಾಗಿ ಕಾಣುತ್ತದೆ ಮತ್ತು ಇದು ಮೋಸದ ಪ್ರಯತ್ನವಾಗಿರಬಹುದು.",
        "scam_action": "ಯಾವುದೇ ಲಿಂಕ್ ಕ್ಲಿಕ್ ಮಾಡಬೇಡಿ, QR ಕೋಡ್ ಸ್ಕ್ಯಾನ್ ಮಾಡಬೇಡಿ ಅಥವಾ ವೈಯಕ್ತಿಕ ಮಾಹಿತಿ ಹಂಚಿಕೊಳ್ಳಬೇಡಿ.",
        "medicine_explanation": "ಇದು ಔಷಧಿ ಅಥವಾ ವೈದ್ಯರ ಸೂಚನೆಗಳನ್ನು ಒಳಗೊಂಡಿರುವಂತೆ ಕಾಣುತ್ತದೆ.",
        "medicine_action": "ವೈದ್ಯರ ಸಲಹೆಯನ್ನು ಎಚ್ಚರಿಕೆಯಿಂದ ಪಾಲಿಸಿ.",
        "bill_explanation": "ಇದು ಬಿಲ್ ಅಥವಾ ಪಾವತಿ ಸಂಬಂಧಿತ ದಾಖಲೆಯಾಗಿ ಕಾಣುತ್ತದೆ.",
        "bill_action": "ಮೊತ್ತ ಮತ್ತು ದಿನಾಂಕವನ್ನು ಎಚ್ಚರಿಕೆಯಿಂದ ಪರಿಶೀಲಿಸಿ.",
        "delivery_explanation": "ಇದು ಡೆಲಿವರಿ ಅಥವಾ ಶಿಪ್‌ಮೆಂಟ್ ಅಪ್‌ಡೇಟ್ ಆಗಿ ಕಾಣುತ್ತದೆ.",
        "delivery_action": "ಅಗತ್ಯವಿದ್ದರೆ ಪ್ಯಾಕೇಜ್ ಟ್ರ್ಯಾಕ್ ಮಾಡಿ.",
        "appointment_explanation": "ಇದು ಅಪಾಯಿಂಟ್‌ಮೆಂಟ್ ಅಥವಾ ಭೇಟಿಯ ವೇಳಾಪಟ್ಟಿಯಾಗಿ ಕಾಣುತ್ತದೆ.",
        "appointment_action": "ದಿನಾಂಕ ಮತ್ತು ಸಮಯವನ್ನು ನೆನಪಿಟ್ಟುಕೊಳ್ಳಿ.",
        "invitation_explanation": "ಇದು ಆಹ್ವಾನ ಅಥವಾ ಕಾರ್ಯಕ್ರಮದ ಸೂಚನೆಯಾಗಿ ಕಾಣುತ್ತದೆ.",
        "invitation_action": "ಕಾರ್ಯಕ್ರಮದ ವಿವರಗಳು ಮತ್ತು ಸಮಯವನ್ನು ಪರಿಶೀಲಿಸಿ.",
        "bank_explanation": "ಇದು ಬ್ಯಾಂಕಿಂಗ್ ಅಥವಾ ವ್ಯವಹಾರ ಸಂದೇಶವಾಗಿ ಕಾಣುತ್ತದೆ.",
        "bank_action": "ವ್ಯವಹಾರದ ವಿವರಗಳನ್ನು ಎಚ್ಚರಿಕೆಯಿಂದ ಪರಿಶೀಲಿಸಿ.",
        "shopping_explanation": "ಇದು ಶಾಪಿಂಗ್ ಸಂಬಂಧಿತ ಸಂದೇಶವಾಗಿ ಕಾಣುತ್ತದೆ.",
        "shopping_action": "ಆರ್ಡರ್ ವಿವರಗಳನ್ನು ಪರಿಶೀಲಿಸಿ.",
        "travel_explanation": "ಇದು ಪ್ರಯಾಣ ಅಥವಾ ಟಿಕೆಟ್ ಸಂಬಂಧಿತ ಸಂದೇಶವಾಗಿ ಕಾಣುತ್ತದೆ.",
        "travel_action": "ನಿಮ್ಮ ಪ್ರಯಾಣ ವೇಳಾಪಟ್ಟಿಯನ್ನು ಎಚ್ಚರಿಕೆಯಿಂದ ಪರಿಶೀಲಿಸಿ.",
        "general_explanation": "ಇದು ಸಾಮಾನ್ಯ ಮಾಹಿತಿ ದಾಖಲೆಯಾಗಿದೆ.",
        "general_action": "ದಯವಿಟ್ಟು ಎಚ್ಚರಿಕೆಯಿಂದ ಓದಿ ಅಥವಾ ಖಚಿತವಿಲ್ಲದಿದ್ದರೆ ಕುಟುಂಬದ ಸದಸ್ಯರನ್ನು ಕೇಳಿ.",
        "error_explanation": "AI ಗೆ ಸಂಪರ್ಕಿಸಲು ಆಗಲಿಲ್ಲ. ನಿಮ್ಮ API ಕೀ ಮತ್ತು ಇಂಟರ್ನೆಟ್ ಪರಿಶೀಲಿಸಿ.",
        "error_action": "ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ.",
        "no_input_explanation": "ದಯವಿಟ್ಟು ಯಾವುದಾದರೂ ಇನ್‌ಪುಟ್ ಒದಗಿಸಿ.",
        "no_input_action": "ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ.",
        "stt_error": "ಸ್ಪಷ್ಟವಾಗಿ ಕೇಳಿಸಲಿಲ್ಲ.",
        "stt_retry": "ದಯವಿಟ್ಟು ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ.",
    },
    "Hindi": {
        "scam_explanation": "यह संदेश संदिग्ध लग रहा है और यह धोखाधड़ी का प्रयास हो सकता है।",
        "scam_action": "किसी भी लिंक पर क्लिक न करें, QR कोड स्कैन न करें और कोई भी व्यक्तिगत जानकारी साझा न करें।",
        "medicine_explanation": "यह दवाई या डॉक्टर के निर्देशों से संबंधित दस्तावेज़ लग रहा है।",
        "medicine_action": "डॉक्टर की सलाह का ध्यान से पालन करें।",
        "bill_explanation": "यह बिल या भुगतान से संबंधित दस्तावेज़ लग रहा है।",
        "bill_action": "राशि और देय तिथि को ध्यान से जांचें।",
        "delivery_explanation": "यह डिलीवरी या शिपमेंट अपडेट लग रहा है।",
        "delivery_action": "जरूरत हो तो पैकेज ट्रैक करें।",
        "appointment_explanation": "यह किसी अपॉइंटमेंट या निर्धारित भेंट का संदेश लग रहा है।",
        "appointment_action": "तारीख और समय याद रखें।",
        "invitation_explanation": "यह निमंत्रण या कार्यक्रम की सूचना लग रहा है।",
        "invitation_action": "कार्यक्रम का विवरण और समय जांचें।",
        "bank_explanation": "यह बैंकिंग या लेन-देन संबंधी संदेश लग रहा है।",
        "bank_action": "लेन-देन का विवरण ध्यान से देखें।",
        "shopping_explanation": "यह खरीदारी से संबंधित संदेश लग रहा है।",
        "shopping_action": "ऑर्डर का विवरण जांचें।",
        "travel_explanation": "यह यात्रा या टिकट से संबंधित संदेश लग रहा है।",
        "travel_action": "अपना यात्रा कार्यक्रम ध्यान से देखें।",
        "general_explanation": "यह एक सामान्य जानकारी का दस्तावेज़ है।",
        "general_action": "कृपया ध्यान से पढ़ें या अनिश्चित होने पर परिवार के किसी सदस्य से पूछें।",
        "error_explanation": "AI से कनेक्ट नहीं हो पाया। अपनी API key और इंटरनेट जांचें।",
        "error_action": "कृपया फिर से प्रयास करें।",
        "no_input_explanation": "कृपया कोई इनपुट दें।",
        "no_input_action": "फिर से प्रयास करें।",
        "stt_error": "स्पष्ट रूप से सुनाई नहीं दिया।",
        "stt_retry": "कृपया फिर से प्रयास करें।",
    }
}

def L(lang, key):
    """Get label in the selected language, fallback to English."""
    return LANG_LABELS.get(lang, LANG_LABELS["English"]).get(key, LANG_LABELS["English"].get(key, ""))

def classify_text(text, lang="English"):
    text = text.lower().strip()

    if re.search(
        r'http|https|www\.|bit\.ly|tinyurl|'
        r'otp|password|verify|verification|'
        r'click here|urgent|blocked|suspended|'
        r'account locked|account blocked|'
        r'bank account|kyc|pan card|aadhaar|'
        r'claim now|winner|lottery|reward|'
        r'free gift|update now|pay immediately|'
        r'limited time|expire today|'
        r'sbi|hdfc|icici|axis bank|'
        r'upi collect|scan qr|refund link',
        text
    ):
        return {
            "category": "Scam",
            "is_scam": True,
            "risk": 95,
            "explanation": L(lang, "scam_explanation"),
            "action": L(lang, "scam_action"),
        }

    elif (
        any(k in text for k in [
            "tablet", "capsule", "prescription", "doctor prescribed",
            "after food", "before food", "take once daily", "take twice daily", "rx"
        ])
        or re.search(r'\b\d+\s?(mg|ml)\b', text)
    ):
        return {
            "category": "Medicine",
            "is_scam": False,
            "risk": 20,
            "explanation": L(lang, "medicine_explanation"),
            "action": L(lang, "medicine_action"),
        }

    elif (
        any(k in text for k in [
            "invoice", "bill", "amount due", "due date", "payment receipt",
            "electricity bill", "water bill", "gas bill", "tax invoice", "gst"
        ])
        or ("₹" in text and "due" in text)
    ):
        return {
            "category": "Bill",
            "is_scam": False,
            "risk": 40,
            "explanation": L(lang, "bill_explanation"),
            "action": L(lang, "bill_action"),
        }

    elif any(k in text for k in [
        "delivery", "order shipped", "tracking id", "courier",
        "arriving today", "package", "amazon", "flipkart", "bluedart", "delhivery"
    ]):
        return {
            "category": "Delivery",
            "is_scam": False,
            "risk": 10,
            "explanation": L(lang, "delivery_explanation"),
            "action": L(lang, "delivery_action"),
        }

    elif any(k in text for k in [
        "appointment", "doctor visit", "consultation", "meeting",
        "scheduled", "clinic", "hospital", "checkup"
    ]):
        return {
            "category": "Appointment",
            "is_scam": False,
            "risk": 10,
            "explanation": L(lang, "appointment_explanation"),
            "action": L(lang, "appointment_action"),
        }

    elif any(k in text for k in [
        "wedding", "marriage", "invitation", "ceremony", "reception",
        "festival", "celebration", "party", "birthday", "anniversary",
        "get together", "event"
    ]):
        return {
            "category": "Invitation",
            "is_scam": False,
            "risk": 5,
            "explanation": L(lang, "invitation_explanation"),
            "action": L(lang, "invitation_action"),
        }

    elif any(k in text for k in [
        "credited", "debited", "withdrawal", "deposit", "upi",
        "balance", "transaction", "account statement"
    ]):
        return {
            "category": "Bank",
            "is_scam": False,
            "risk": 25,
            "explanation": L(lang, "bank_explanation"),
            "action": L(lang, "bank_action"),
        }

    elif any(k in text for k in [
        "shopping", "purchase", "cart", "item ordered", "buy now"
    ]):
        return {
            "category": "Shopping",
            "is_scam": False,
            "risk": 15,
            "explanation": L(lang, "shopping_explanation"),
            "action": L(lang, "shopping_action"),
        }

    elif any(k in text for k in [
        "flight", "train", "boarding", "ticket", "pnr", "departure", "arrival"
    ]):
        return {
            "category": "Travel",
            "is_scam": False,
            "risk": 15,
            "explanation": L(lang, "travel_explanation"),
            "action": L(lang, "travel_action"),
        }

    else:
        return {
            "category": "General",
            "is_scam": False,
            "risk": 25,
            "explanation": L(lang, "general_explanation"),
            "action": L(lang, "general_action"),
        }

def analyze_with_gemini(img, lang="English"):
    model = genai.GenerativeModel("gemini-1.5-flash-003")
    try:
        res = model.generate_content(
            [get_prompt(lang), img],
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=350,
                temperature=0.1,
            )
        )
        raw = res.text.strip()
        print("Gemini raw:", raw[:300])
        raw = raw.replace("```json", "").replace("```", "").strip()
        s = raw.find("{")
        e = raw.rfind("}") + 1
        if s == -1:
            raise ValueError("No JSON in response")
        data = json.loads(raw[s:e])
        for k, v in [("category", "General"), ("is_scam", False), ("risk", 20),
                     ("explanation", L(lang, "general_explanation")),
                     ("action", L(lang, "general_action")),
                     ("date", ""), ("time", "")]:
            data.setdefault(k, v)
        return data
    except Exception as e:
        print("Gemini Vision error:", type(e).__name__, e)
        return {
            "category": "Error", "is_scam": False, "risk": 0,
            "explanation": L(lang, "error_explanation"),
            "action": L(lang, "error_action"),
            "date": "", "time": ""
        }

def build_result_html(cat, risk, is_scam):
    risk = int(risk)
    if is_scam or risk >= 70:
        bg, border, icon, label, bar_c = "#FFF3F3", "#FFCDD2", "⚠️", "Danger — Possible Scam", "#E53935"
        txt = "#B71C1C"
    elif risk >= 35:
        bg, border, icon, label, bar_c = "#FFFDE7", "#FFF176", "⚡", "Needs Your Attention", "#F57F17"
        txt = "#E65100"
    else:
        bg, border, icon, label, bar_c = "#F1F8E9", "#C5E1A5", "✅", "Looks Safe", "#2E7D32"
        txt = "#1B5E20"

    scam_block = ""
    if is_scam:
        scam_block = f"""
        <div style="background:#FFEBEE;border:2px solid #EF9A9A;border-radius:16px;
                    padding:18px;margin-top:14px;color:#B71C1C;font-size:18px;
                    font-weight:700;line-height:1.6;">
          ⚠️ <span style="color:#B71C1C;font-weight:900;">
                WARNING: This may be a SCAM!
                </span><br>
          Do NOT click any links.<br>
          Do NOT call any numbers.<br>
          Show this to a family member NOW.
        </div>"""

    return f"""
    <div style="background:{bg};border:2px solid {border};border-radius:20px;padding:22px;margin-bottom:14px;">
      <div style="font-size:13px;font-weight:800;text-transform:uppercase;
                  letter-spacing:.1em;color:{txt};margin-bottom:6px;">{icon} {label}</div>
      <div style="font-size:28px;font-weight:900;color:#1A237E;margin-bottom:16px;">{cat}</div>
      <div style="height:12px;border-radius:6px;background:rgba(0,0,0,.1);
                  overflow:hidden;margin-bottom:8px;">
        <div style="height:100%;width:{risk}%;background:{bar_c};border-radius:6px;"></div>
      </div>
      <div style="font-size:15px;font-weight:700;color:{txt};">Risk: {risk}%</div>
      {scam_block}
    </div>"""

LOADING_HTML = """
<div style="background:#E8EAF6;border:2px solid #9FA8DA;border-radius:20px;
            padding:30px 20px;text-align:center;margin-bottom:14px;">
  <div style="font-size:40px;margin-bottom:12px;">🔍</div>
  <div style="font-size:22px;font-weight:900;color:#1A237E;margin-bottom:8px;">
    Reading Your Document...
  </div>
  <div style="font-size:17px;color:#5C6BC0;font-weight:600;line-height:1.6;">
    Please wait a moment.<br>This usually takes 5 to 10 seconds.
  </div>
  <div style="margin-top:18px;height:8px;border-radius:4px;background:#C5CAE9;overflow:hidden;">
    <div style="height:100%;width:100%;
                background:linear-gradient(90deg,#3949AB,#1565C0,#3949AB);
                background-size:200%;animation:shimmer 1.5s linear infinite;">
    </div>
  </div>
  <style>@keyframes shimmer{0%{background-position:200% 0}100%{background-position:-200% 0}}</style>
</div>
"""

def show_loading():
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        LOADING_HTML,
        "Analyzing your document...",
        "Please wait..."
    )

def extract_date_time(text):
    text = text.lower()
    date = ""
    time = ""

    d1 = re.search(r'(\d{2})[-/](\d{2})[-/](\d{4})', text)
    if d1:
        d, m, y = d1.groups()
        date = f"{y}-{m}-{d}"

    d2 = re.search(r'(\d{1,2})(st|nd|rd|th)?\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+(\d{4})', text)
    if d2:
        day = d2.group(1)
        month_str = d2.group(3)
        year = d2.group(4)
        try:
            month = datetime.strptime(month_str[:3], "%b").month
            date = f"{int(day):02d}/{month:02d}/{year}"
        except:
            pass

    d3 = re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+(\d{1,2}),?\s*(\d{4})', text)
    if d3:
        month_str = d3.group(1)
        day = d3.group(2)
        year = d3.group(3)
        try:
            month = datetime.strptime(month_str[:3], "%b").month
            date = f"{int(day):02d}/{month:02d}/{year}"
        except:
            pass

    t = re.search(r'(\d{1,2})([:.](\d{2}))?\s*(am|pm)?', text)
    if t:
        hour = int(t.group(1))
        minute = t.group(3) if t.group(3) else "00"
        period = t.group(4)
        if period:
            period = period.lower()
            if period == "pm" and hour != 12:
                hour += 12
            if period == "am" and hour == 12:
                hour = 0
        time = f"{hour:02d}:{minute}"

    return date, time

def extract_date_from_invitation(text):
    text = text.lower()
    patterns = [
        r'(\d{1,2})[-/](\d{1,2})[-/](\d{4})',
        r'(\d{1,2})(st|nd|rd|th)?\s+(march|april|may|june|july|august|september|october|november|december)\s+(\d{4})',
        r'(march|april|may|june|july|august)\s+(\d{1,2}),?\s*(\d{4})'
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            try:
                if len(m.groups()) == 3:
                    d, mth, y = m.groups()
                else:
                    d = m.group(1)
                    mth = m.group(3)
                    y = m.group(4)
                month = datetime.strptime(mth[:3], "%b").month
                return f"{y}-{month:02d}-{int(d):02d}"
            except:
                pass
    return ""

def extract_numbers_date(text):
    m = re.search(r'(\d{1,2})\D*(\d{1,2})\D*(\d{4})', text)
    if m:
        d, mth, y = m.groups()
        if 1 <= int(d) <= 31 and 1 <= int(mth) <= 12:
            return f"{int(d):02d}/{mth:02d}/{y}"
    return ""

# ── NEW: spoken date patterns like "fifth of May 2026", "May fifth 2026" ──
SPOKEN_MONTHS = {
    "january":1,"february":2,"march":3,"april":4,"may":5,"june":6,
    "july":7,"august":8,"september":9,"october":10,"november":11,"december":12,
    # short forms
    "jan":1,"feb":2,"mar":3,"apr":4,"jun":6,"jul":7,"aug":8,
    "sep":9,"sept":9,"oct":10,"nov":11,"dec":12,
}
SPOKEN_ORDINALS = {
    "first":"1","second":"2","third":"3","fourth":"4","fifth":"5",
    "sixth":"6","seventh":"7","eighth":"8","ninth":"9","tenth":"10",
    "eleventh":"11","twelfth":"12","thirteenth":"13","fourteenth":"14",
    "fifteenth":"15","sixteenth":"16","seventeenth":"17","eighteenth":"18",
    "nineteenth":"19","twentieth":"20","twenty first":"21","twenty second":"22",
    "twenty third":"23","twenty fourth":"24","twenty fifth":"25",
    "twenty sixth":"26","twenty seventh":"27","twenty eighth":"28",
    "twenty ninth":"29","thirtieth":"30","thirty first":"31",
}

def extract_spoken_date(text):
    """
    Handles spoken formats that STT produces, e.g.:
      "fifth of May 2026"
      "May fifth 2026"
      "on the 5th of May 2026"
      "appointment on 5 May 2026"
      "reminder for tomorrow" (skipped — no year)
    Returns dd/mm/yyyy string or "".
    """
    text = text.lower()

    # replace ordinal words with digits  ("twenty fifth" → "25th")
    for word, digit in sorted(SPOKEN_ORDINALS.items(), key=lambda x: -len(x[0])):
        text = text.replace(word, digit + "th")

    # pattern: <day>(st/nd/rd/th)? (of)? <month> <year>
    p1 = re.search(
        r'(\d{1,2})(?:st|nd|rd|th)?\s+(?:of\s+)?'
        r'(january|february|march|april|may|june|july|august|september|october|november|december'
        r'|jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec)'
        r'\s+(\d{4})',
        text
    )
    if p1:
        day, mon, year = p1.group(1), p1.group(2), p1.group(3)
        m = SPOKEN_MONTHS.get(mon)
        if m:
            return f"{int(day):02d}/{m:02d}/{year}"

    # pattern: <month> <day>(st/nd/rd/th)? <year>
    p2 = re.search(
        r'(january|february|march|april|may|june|july|august|september|october|november|december'
        r'|jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec)'
        r'\s+(\d{1,2})(?:st|nd|rd|th)?\s+(\d{4})',
        text
    )
    if p2:
        mon, day, year = p2.group(1), p2.group(2), p2.group(3)
        m = SPOKEN_MONTHS.get(mon)
        if m:
            return f"{int(day):02d}/{m:02d}/{year}"

    return ""

def extract_spoken_time(text):
    """
    Handles spoken time like:
      "at 6 pm", "at 6:30 pm", "at 18:30", "at six thirty"
    Returns HH:MM string or "".
    """
    text = text.lower()

    # word numbers for hours
    word_hours = {
        "one":"1","two":"2","three":"3","four":"4","five":"5","six":"6",
        "seven":"7","eight":"8","nine":"9","ten":"10","eleven":"11","twelve":"12",
    }
    for w, d in word_hours.items():
        text = re.sub(r'\b' + w + r'\b', d, text)

    # "thirty" → ":30" when used after an hour
    text = re.sub(r'(\d{1,2})\s+thirty\b', r'\1:30', text)
    text = re.sub(r'(\d{1,2})\s+fifteen\b', r'\1:15', text)
    text = re.sub(r'(\d{1,2})\s+forty\s*five\b', r'\1:45', text)
    text = re.sub(r'(\d{1,2})\s+o\'?\s*clock\b', r'\1:00', text)

    t = re.search(r'(\d{1,2})(?::(\d{2}))?\s*(am|pm)', text)
    if t:
        hour = int(t.group(1))
        minute = t.group(2) if t.group(2) else "00"
        period = t.group(3).lower()
        if period == "pm" and hour != 12:
            hour += 12
        if period == "am" and hour == 12:
            hour = 0
        return f"{hour:02d}:{minute}"

    # 24-h fallback
    t2 = re.search(r'\bat\s+(\d{1,2}):(\d{2})\b', text)
    if t2:
        return f"{int(t2.group(1)):02d}:{t2.group(2)}"

    return ""

def full_date_extract(text):
    """
    Run all date extractors in order, return first non-empty result.
    Same pipeline now used for VOICE, TEXT, and IMAGE paths.
    """
    date, time = extract_date_time(text)

    # spoken-date extractor (catches "fifth of May 2026" etc.)
    if not date:
        date = extract_spoken_date(text)

    if not date:
        date = extract_date_from_invitation(text)

    if not date:
        date = extract_numbers_date(text)

    # spoken-time extractor
    if not time:
        time = extract_spoken_time(text)

    return date, time

import concurrent.futures

def safe_ocr(img):
    def task():
        return pytesseract.image_to_string(img, config="--psm 6")
    try:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(task)
            return future.result(timeout=3)
    except:
        return ""

def run(image, typed_text, voice, lang):
    lang = lang or "English"
    try:
        # ================= VOICE =================
        if voice:
            text = speech_to_text(voice)
            if not text:
                return (
                    build_result_html("Error", 0, False),
                    L(lang, "stt_error"),
                    L(lang, "stt_retry"),
                    "", ""
                )
            t = text.lower()
            data = classify_text(t, lang)
            # ✅ now uses full pipeline — same as image path
            date, time = full_date_extract(t)
            html = build_result_html(data["category"], data["risk"], data["is_scam"])
            return html, data["explanation"], data["action"], date, time if time else ""

        # ================= TEXT =================
        if typed_text and typed_text.strip():
            text = typed_text.strip().lower()
            data = classify_text(text, lang)
            # ✅ full pipeline for typed text too
            date, time = full_date_extract(text)
            html = build_result_html(data["category"], data["risk"], data["is_scam"])
            return html, data["explanation"], data["action"], date, time if time else ""

        # ================= IMAGE =================
        if image is not None:
            try:
                with concurrent.futures.ThreadPoolExecutor() as executor:
                    future = executor.submit(ocr_image_text, image)
                    text = future.result(timeout=3)
            except:
                text = ocr_image_text(image)

            text = (text or "").lower().strip()

            if len(text) < 10:
                try:
                    raw = pytesseract.image_to_string(image, config="--psm 6 digits")
                    text += " " + raw.lower()
                except:
                    pass

            # ✅ full pipeline
            date, time = full_date_extract(text)
            if not time:
                time = ""

            data = classify_text(text, lang)
            html = build_result_html(data["category"], data["risk"], data["is_scam"])
            return html, data["explanation"], data["action"], date, time

        # ================= NO INPUT =================
        return (
            build_result_html("No Input", 0, False),
            L(lang, "no_input_explanation"),
            L(lang, "no_input_action"),
            "", ""
        )

    except Exception as e:
        print("ERROR:", e)
        return (
            build_result_html("Error", 0, False),
            L(lang, "error_explanation"),
            L(lang, "error_action"),
            "", ""
        )

def add_reminder(result_html, date, time_str):
    if not date:
        return """
        <div style="
            background:#FFEBEE;
            border:2px solid #EF9A9A;
            border-radius:16px;
            padding:18px;
            color:#B71C1C;
            font-size:18px;
            font-weight:800;">
            ⚠️ No date found.
        </div>
        """
    title = "Reminder"
    for c in ["Appointment","Invitation","Medicine","Bill","Delivery",
              "Bank","Travel","Shopping","Scam","General"]:
        if c.lower() in result_html.lower():
            title = c
            break
    if not time_str:
        time_str = "09:00"
    return make_calendar_html(title, date, time_str)

# ── LANGUAGE → SPEECH SYNTHESIS BCP-47 LOCALE MAPPING ───────────
LANG_LOCALE = {
    "English": "en-IN",
    "Kannada": "kn-IN",
    "Hindi":   "hi-IN",
}

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@600;700;800;900&display=swap');

body { background: #E8EAF6 !important; margin: 0 !important; }

.gradio-container {
    max-width: 440px !important;
    min-height: 100vh !important;
    margin: 0 auto !important;
    background: #FAFAFA !important;
    font-family: 'Nunito', sans-serif !important;
    padding: 0 !important;
    box-shadow: 0 0 40px rgba(0,0,0,.15) !important;
}
footer { display: none !important; }
button { font-family: 'Nunito', sans-serif !important; cursor: pointer !important; border: none !important; }

#go_btn {
    background: #2E7D32 !important;
    color: #fff !important;
    font-size: 22px !important;
    font-weight: 900 !important;
    border-radius: 18px !important;
    padding: 20px !important;
    width: 100% !important;
    box-shadow: 0 6px 20px rgba(46,125,50,.35) !important;
    margin: 16px 0 !important;
    letter-spacing: .02em !important;
}
#go_btn:hover { background: #1B5E20 !important; }

#back_btn {
    background: #E8EAF6 !important;
    color: #3949AB !important;
    font-size: 17px !important;
    font-weight: 800 !important;
    border-radius: 12px !important;
    padding: 12px 20px !important;
    border: 2px solid #9FA8DA !important;
}
#back_btn:hover { background: #C5CAE9 !important; }

#sp_btn {
    background: #1565C0 !important;
    color: #fff !important;
    font-size: 20px !important;
    font-weight: 800 !important;
    border-radius: 16px !important;
    padding: 18px !important;
    width: 100% !important;
    margin-top: 6px !important;
    box-shadow: 0 6px 18px rgba(21,101,192,.35) !important;
}
#sp_btn:hover { background: #0D47A1 !important; }

#cal_btn {
    background: #2E7D32 !important;
    color: #fff !important;
    font-size: 19px !important;
    font-weight: 800 !important;
    border-radius: 14px !important;
    padding: 16px !important;
    width: 100% !important;
    margin-top: 10px !important;
    box-shadow: 0 4px 14px rgba(46,125,50,.3) !important;
}
#cal_btn:hover { background: #1B5E20 !important; }

#img_in {
    background: #F3F4F6 !important;
    border: 3px dashed #9FA8DA !important;
    border-radius: 18px !important;
}

#txt_in textarea {
    background: #F3F4F6 !important;
    color: #1A237E !important;
    border: 2px solid #9FA8DA !important;
    border-radius: 14px !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 18px !important;
    font-weight: 600 !important;
    padding: 14px 16px !important;
    line-height: 1.6 !important;
}
#txt_in textarea::placeholder { color: #9E9E9E !important; }
#txt_in label { color: #3949AB !important; font-size: 15px !important; font-weight: 800 !important; }

#aud_in {
    background: #F3F4F6 !important;
    border-radius: 14px !important;
    border: 2px solid #9FA8DA !important;
    padding: 10px !important;
}
#aud_in label { color: #6A1B9A !important; font-size: 15px !important; font-weight: 800 !important; }

#expl_out textarea, #act_out textarea {
    background: #F8F9FA !important;
    color: #1A237E !important;
    border: 2px solid #C5CAE9 !important;
    border-radius: 16px !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 19px !important;
    font-weight: 600 !important;
    line-height: 1.8 !important;
    padding: 16px !important;
}
#expl_out label { color: #2E7D32 !important; font-size: 16px !important; font-weight: 800 !important; }
#act_out  label { color: #1565C0 !important; font-size: 16px !important; font-weight: 800 !important; }

#lang_sel label { color: #6A1B9A !important; font-size: 15px !important; font-weight: 800 !important; }
#lang_sel select {
    background: #F3F4F6 !important;
    color: #1A237E !important;
    border: 2px solid #9FA8DA !important;
    border-radius: 12px !important;
    font-size: 18px !important;
    font-weight: 700 !important;
    padding: 10px 14px !important;
}

#cal_date, #cal_time {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    overflow: hidden !important;
}
#cal_date > div, #cal_time > div {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    overflow: hidden !important;
}
#cal_date textarea, #cal_time textarea,
#cal_date input,  #cal_time input {
    background: #FFFDF8 !important;
    color: #3E2723 !important;
    border: 2px solid #FFD59E !important;
    border-radius: 16px !important;
    font-size: 18px !important;
    font-weight: 700 !important;
    padding: 14px 16px !important;
    box-shadow: 0 2px 8px rgba(255,183,77,0.10) !important;
    overflow: hidden !important;
}
#cal_date textarea:focus, #cal_time textarea:focus,
#cal_date input:focus,  #cal_time input:focus {
    border: 2px solid #FFB74D !important;
    box-shadow: 0 0 0 4px rgba(255,183,77,0.18) !important;
    outline: none !important;
}
#cal_date label, #cal_time label {
    color: #E65100 !important;
    font-size: 15px !important;
    font-weight: 900 !important;
}
#cal_date textarea, #cal_time textarea,
#cal_date input,  #cal_time input {
    white-space: nowrap !important;
    overflow: hidden !important;
    resize: none !important;
    height: 52px !important;
}
.padded { padding: 0 18px; }
"""

SPEAK_JS = """
() => {
    const expl = document.querySelector('#expl_out textarea');
    const act  = document.querySelector('#act_out  textarea');
    const t    = (expl ? expl.value : '') + '. ' + (act ? act.value : '');
    if (!t.trim() || t.trim() === '.') { alert('Please analyze a document first.'); return; }

    const langSpan = document.getElementById('current_lang_val');
    const langMap  = { 'English': 'en-IN', 'Kannada': 'kn-IN', 'Hindi': 'hi-IN' };
    const locale   = langSpan ? (langMap[langSpan.textContent.trim()] || 'en-IN') : 'en-IN';

    window.speechSynthesis.cancel();
    const u = new SpeechSynthesisUtterance(t);
    u.rate   = 0.80;
    u.pitch  = 1.0;
    u.volume = 1;
    u.lang   = locale;
    window.speechSynthesis.speak(u);
}
"""

TOPBAR = '<div style="height:8px;background:linear-gradient(90deg,#1A237E,#3949AB,#1565C0);"></div>'
HOME   = '<div style="height:36px;display:flex;align-items:center;justify-content:center;background:#FAFAFA;"><div style="width:140px;height:5px;background:#C5CAE9;border-radius:3px;"></div></div>'

with gr.Blocks(css=CSS, title="ElderAssist AI") as demo:

    lang_state = gr.State("English")

    # ── INPUT PAGE ──────────────────────────────────────────────
    with gr.Column(visible=True) as pg_in:
        gr.HTML(TOPBAR)
        gr.HTML("""
        <div style="padding:24px 22px 16px;background:#1A237E;">
          <div style="font-size:13px;font-weight:700;color:#9FA8DA;
                      letter-spacing:.1em;text-transform:uppercase;margin-bottom:6px;">
            Your Document Helper
          </div>
          <h1 style="font-size:30px;font-weight:900;color:#fff;margin:0 0 4px;line-height:1.2;">
            👵 ElderAssist AI
          </h1>
          <p style="font-size:16px;color:#C5CAE9;margin:0;font-weight:600;">
            Scan any document — we'll explain it simply
          </p>
        </div>
        """)

        with gr.Column(elem_classes="padded"):

            gr.HTML('<p style="font-size:15px;font-weight:800;color:#6A1B9A;margin:14px 0 6px;">🌐 Choose Language / ಭಾಷೆ ಆಯ್ಕೆ ಮಾಡಿ / भाषा चुनें</p>')
            lang_sel = gr.Dropdown(
                choices=["English", "Kannada", "Hindi"],
                value="English",
                label="",
                elem_id="lang_sel",
            )

            gr.HTML('<p style="font-size:15px;font-weight:800;color:#2E7D32;margin:10px 0 6px;">📷 Take or Upload a Photo</p>')
            img_in = gr.Image(type="pil", label="", sources=["upload", "webcam"], height=210, elem_id="img_in")

            gr.HTML('<p style="font-size:15px;font-weight:800;color:#3949AB;margin:14px 0 6px;">✏️ Or Type Your Message</p>')
            txt_in = gr.Textbox(label="", placeholder="Type or paste your message here...", lines=3, elem_id="txt_in")

            gr.HTML('<p style="font-size:15px;font-weight:800;color:#6A1B9A;margin:14px 0 6px;">🎤 Or Record Your Voice</p>')
            aud_in = gr.Audio(sources=["microphone"], type="filepath", label="Tap to record", elem_id="aud_in")

            go_btn = gr.Button("🔍  Analyze Now", elem_id="go_btn")

        gr.HTML(HOME)

    # ── RESULT PAGE ─────────────────────────────────────────────
    with gr.Column(visible=False) as pg_out:
        gr.HTML(TOPBAR)
        gr.HTML("""
        <div style="padding:20px 22px 14px;background:#1A237E;">
          <div style="font-size:13px;font-weight:700;color:#9FA8DA;
                      letter-spacing:.1em;text-transform:uppercase;">Analysis Result</div>
          <h2 style="font-size:26px;font-weight:900;color:#fff;margin:4px 0 0;">
            Here Is What We Found
          </h2>
        </div>
        """)

        with gr.Column(elem_classes="padded"):
            with gr.Row():
                back_btn = gr.Button("← Go Back", elem_id="back_btn", scale=0)
                gr.HTML('<div></div>')

            result_html = gr.HTML()

            expl_out = gr.Textbox(label="📖  What This Document Says", lines=4, interactive=False, elem_id="expl_out")
            act_out  = gr.Textbox(label="✅  What You Should Do",       lines=3, interactive=False, elem_id="act_out")

            lang_display = gr.HTML('<span id="current_lang_val" style="display:none;">English</span>')

            sp_btn = gr.Button("🔊  Read Aloud — Tap to Listen", elem_id="sp_btn")

            gr.HTML("""
            <div style="margin-top:22px;background:#FFF8E1;border:2px solid #FFE082;
                        border-radius:20px;padding:20px;">
              <div style="font-size:16px;font-weight:900;color:#E65100;text-align:center;
                          text-transform:uppercase;letter-spacing:.08em;margin-bottom:6px;">
                🗓️ Set a Reminder
              </div>
            </div>
            """)

            with gr.Row():
                cal_date = gr.Textbox(
                    label="📆 Date", placeholder="01/05/2026",
                    elem_id="cal_date", lines=1, max_lines=1,
                    text_align="center", scale=1
                )
                cal_time = gr.Textbox(
                    label="⏰ Time", placeholder="16:00",
                    elem_id="cal_time", lines=1, max_lines=1,
                    text_align="center", scale=1
                )
            cal_btn = gr.Button("📅  Save to Calendar", elem_id="cal_btn", size="lg")
            gr.HTML("</div>")
            cal_out = gr.HTML()

        gr.HTML(HOME)

    # ── Events ──────────────────────────────────────────────────

    def update_lang_display(lang):
        return f'<span id="current_lang_val" style="display:none;">{lang}</span>'

    go_btn.click(
        fn=show_loading,
        inputs=None,
        outputs=[pg_in, pg_out, result_html, expl_out, act_out],
    ).then(
        fn=run,
        inputs=[img_in, txt_in, aud_in, lang_sel],
        outputs=[result_html, expl_out, act_out, cal_date, cal_time],
    ).then(
        fn=update_lang_display,
        inputs=[lang_sel],
        outputs=[lang_display],
    )

    def reset_all():
        return (
            gr.update(visible=True),
            gr.update(visible=False),
            None, "", None,
            "", "", "",
            "", "", "",
            '<span id="current_lang_val" style="display:none;">English</span>'
        )

    back_btn.click(
        fn=reset_all,
        inputs=None,
        outputs=[
            pg_in, pg_out,
            img_in, txt_in, aud_in,
            result_html, expl_out, act_out,
            cal_date, cal_time, cal_out,
            lang_display
        ],
    )

    sp_btn.click(fn=None, inputs=None, outputs=None, js=SPEAK_JS)

    cal_btn.click(
        fn=add_reminder,
        inputs=[result_html, cal_date, cal_time],
        outputs=[cal_out]
    )

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_11257/2260719434.py:1031: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="ElderAssist AI") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d726dbf5d800dc8dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
